# ワークフロー自動化 Airflow スタイル

##　目的
- Airflowの「DAG(有効非巡回グラフ)の考え方を学ぶ
- タスクの依存関係と実行順序を設計する
- Pythonでワークフローエンジンを実装する
- エラーハンドリング、リトライ、ログ記録を自動化する
- Day17 で本物のAirflowに移行するための基礎を作る

## Airflowとは
Apache Airflowは「データパイプラインのスケジュール実行。監視」ツール

## 革新概念: DAG(Directed Acyclic Graph)
-Directed:タスクに方向（順序）がある(A→B→C)
- Acyclic:循環しない(A→B→A)はだめ
- Graph:タスク同士がつながったネットワーク

## 今回のパイプラインのDAG


In [1]:
# ライブラリのインポート
import sqlite3
import pandas as pd
import numpy as np
from scipy import stats
from datetime import datetime
import time
import traceback

print("✓ ライブラリのインポート完了")
print(f"  実行時刻: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✓ ライブラリのインポート完了
  実行時刻: 2026-04-09 07:48:38


## 1.ワークフローエンジンの実装

### Airflowだとこう書く:
'''python
# AirflowのDAG定義
dag = DAG('nikkei225_etl', schedule_interval='0 9 * * 1-5')

task_extract >> task_transform >> [task_quality, task_load] >> task_report
'''

### Pythonで同じ仕組みを作る：
-各タスクを関数として定義
-実行時間、成功/失敗、エラー内容をログに記録
-失敗時のリトライ機能
-依存関係に従った実行順序の整理


In [2]:
# ワークフローエンジン

class TaskResult:
    """タスクの実行結果を保持"""
    def __init__(self, task_name, status, duration, message="", data=None):
        self.task_name = task_name
        self.status = status
        self.duration = duration
        self.message = message
        self.data = data

class WorkflowEngine:
    """Airflowスタイルのワークフローエンジン""" 

    def __init__(self, name):
        self.name = name
        self.results = []
        self.start_time = None
    
    def run_task(self, task_name, func, retries=1, *args, **kwargs):
        """ラスクを実行（リトライ機能付き）"""
        print(f"\n{'─' * 50}")
        print(f"[TASK] {task_name}")
        print(f"{'─' * 50}")
        
        for attempt in range(1, retries + 1):
            start = time.time()
            try:
                result_data = func(*args, **kwargs)
                duration = time.time() - start
                
                result = TaskResult(task_name, "SUCCESS", duration, data=result_data)
                self.results.append(result)
                
                print(f"  ✓ 完了（{duration:.2f}秒）")
                return result
                
            except Exception as e:
                duration = time.time() - start
                if attempt < retries:
                    print(f"  ⚠ 試行 {attempt}/{retries} 失敗: {e}")
                    print(f"    → リトライします...")
                    time.sleep(1)
                else:
                    result = TaskResult(task_name, "FAILED", duration, message=str(e))
                    self.results.append(result)
                    print(f"  ✗ 失敗: {e}")
                    return result
    
    def run_parallel(self, tasks):
        """複数タスクを実行（本来は並列、ここでは順次実行で模擬）"""
        print(f"\n{'━' * 50}")
        print(f"[PARALLEL] {len(tasks)} タスクを実行")
        print(f"{'━' * 50}")
        
        results = []
        for task_name, func, kwargs in tasks:
            result = self.run_task(task_name, func, **kwargs)
            results.append(result)
        return results
    
    def summary(self):
        """実行結果のサマリーを表示"""
        total_time = sum(r.duration for r in self.results)
        
        print(f"\n{'═' * 60}")
        print(f"ワークフロー実行結果: {self.name}")
        print(f"{'═' * 60}")
        print(f"{'タスク名':<30} {'状態':<10} {'時間':>8}")
        print(f"{'─' * 60}")
        
        for r in self.results:
            status_icon = "✓" if r.status == "SUCCESS" else "✗"
            print(f"{status_icon} {r.task_name:<28} {r.status:<10} {r.duration:>6.2f}秒")
        
        print(f"{'─' * 60}")
        success = sum(1 for r in self.results if r.status == "SUCCESS")
        total = len(self.results)
        print(f"合計: {success}/{total} 成功 | 総実行時間: {total_time:.2f}秒")
        print(f"{'═' * 60}")
        
        return success == total

print("✓ WorkflowEngine クラス定義完了")

✓ WorkflowEngine クラス定義完了


## 2.各タスクの定義

DAGの各ノード（タスク）を関数として定義する
AirflowではPythonOperatorでラップするが、ここでは普通の関数。

In [3]:
# タスク定義
import yfinance as yf

def task_extract():
    """Extract: yfinanceからデータ取得"""
    print(" yfinanceからデータ取得中")
    nikkei = yf.download('^N225', start='2019-01-01')
    nikkei.columns = nikkei.columns.get_level_values(0)
    
    row_count = len(nikkei)
    assert row_count > 0, "データ取得に失敗（0行）"
    
    print(f"  {row_count} 行取得")
    print(f"  期間: {nikkei.index[0].strftime('%Y-%m-%d')} 〜 {nikkei.index[-1].strftime('%Y-%m-%d')}")
    return nikkei


def task_transform(df):
    """Transform: SQLite でデータ加工（dbt スタイルの層構造）"""
    conn = sqlite3.connect('nikkei225.db')
    
    # --- Raw 層 ---
    df_sql = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
    df_sql.index = df_sql.index.strftime('%Y-%m-%d')
    df_sql.index.name = 'date'
    df_sql.to_sql('raw_nikkei225', conn, if_exists='replace')
    print(f"  raw_nikkei225: {len(df)} 行")
    
    # --- Staging 層 ---
    conn.execute("DROP TABLE IF EXISTS stg_nikkei225")
    conn.execute("""
        CREATE TABLE stg_nikkei225 AS
        SELECT date,
            ROUND(CAST(Open AS REAL), 2) as open_price,
            ROUND(CAST(High AS REAL), 2) as high_price,
            ROUND(CAST(Low AS REAL), 2) as low_price,
            ROUND(CAST(Close AS REAL), 2) as close_price,
            CAST(Volume AS INTEGER) as volume
        FROM raw_nikkei225
        WHERE Close IS NOT NULL
        ORDER BY date
    """)
    print("  stg_nikkei225: 作成完了")
    
    # --- Intermediate 層 ---
    conn.execute("DROP TABLE IF EXISTS int_daily_metrics")
    conn.execute("""
        CREATE TABLE int_daily_metrics AS
        SELECT date, close_price, volume,
            ROUND((close_price - LAG(close_price) OVER (ORDER BY date))
                / LAG(close_price) OVER (ORDER BY date) * 100, 4) as daily_return,
            ROUND(AVG(close_price) OVER (
                ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW), 2) as ma_20,
            ROUND(AVG(close_price) OVER (
                ORDER BY date ROWS BETWEEN 199 PRECEDING AND CURRENT ROW), 2) as ma_200
        FROM stg_nikkei225
        ORDER BY date
    """)
    print("  int_daily_metrics: 作成完了")
    
    conn.commit()
    conn.close()
    return len(df)


def task_quality_check():
    """Quality: データ品質テスト"""
    conn = sqlite3.connect('nikkei225.db')
    
    tests = [
        ("NULL チェック", "SELECT COUNT(*) FROM stg_nikkei225 WHERE close_price IS NULL"),
        ("重複チェック", "SELECT COUNT(*) FROM (SELECT date, COUNT(*) c FROM stg_nikkei225 GROUP BY date HAVING c > 1)"),
        ("正の値チェック", "SELECT COUNT(*) FROM stg_nikkei225 WHERE close_price <= 0"),
        ("MA20 範囲チェック", """SELECT COUNT(*) FROM int_daily_metrics 
            WHERE ma_20 IS NOT NULL AND (ma_20 > close_price * 2 OR ma_20 < close_price * 0.5)"""),
    ]
    
    passed = 0
    for test_name, query in tests:
        result = pd.read_sql(query, conn).iloc[0, 0]
        status = "PASS" if result == 0 else "FAIL"
        print(f"  {status}: {test_name}")
        if result == 0:
            passed += 1
    
    conn.close()
    assert passed == len(tests), f"品質テスト失敗: {passed}/{len(tests)}"
    print(f"  全 {len(tests)} テスト通過")
    return passed


def task_load_analysis():
    """Load: 統計分析を実行"""
    conn = sqlite3.connect('nikkei225.db')
    
    # 基本統計
    stats_df = pd.read_sql("""
        SELECT COUNT(*) as total,
            MIN(date) as start_date,
            MAX(date) as end_date,
            ROUND(AVG(close_price), 2) as avg_price
        FROM stg_nikkei225
    """, conn)
    print(f"  データ期間: {stats_df['start_date'][0]} 〜 {stats_df['end_date'][0]}")
    print(f"  平均株価: {stats_df['avg_price'][0]:,.2f} 円")
    
    # シグナル生成 + t検定
    signals = pd.read_sql("""
        SELECT date, close_price, ma_20, ma_200,
            CASE WHEN ma_20 > ma_200 
                AND LAG(ma_20) OVER (ORDER BY date) <= LAG(ma_200) OVER (ORDER BY date)
                THEN 'BUY'
                WHEN ma_20 < ma_200 
                AND LAG(ma_20) OVER (ORDER BY date) >= LAG(ma_200) OVER (ORDER BY date)
                THEN 'SELL'
            END as signal
        FROM int_daily_metrics WHERE ma_200 IS NOT NULL
    """, conn)
    
    sig = signals[signals['signal'].notna()]
    buys = sig[sig['signal'] == 'BUY'].reset_index(drop=True)
    sells = sig[sig['signal'] == 'SELL'].reset_index(drop=True)
    
    returns = []
    for _, buy in buys.iterrows():
        future = sells[sells['date'] > buy['date']]
        if len(future) > 0:
            sell = future.iloc[0]
            returns.append((sell['close_price'] - buy['close_price']) / buy['close_price'] * 100)
    
    if returns:
        t_stat, p_value = stats.ttest_1samp(returns, 0)
        print(f"  トレード数: {len(returns)}")
        print(f"  平均リターン: {np.mean(returns):.2f}%")
        print(f"  p値: {p_value:.6f}")
    
    conn.close()
    return {'trades': len(returns), 'p_value': p_value}


def task_report(workflow_results):
    """Report: 実行結果レポートを生成"""
    conn = sqlite3.connect('nikkei225.db')
    
    # レポートテーブル作成
    conn.execute("DROP TABLE IF EXISTS pipeline_log")
    conn.execute("""
        CREATE TABLE pipeline_log (
            run_timestamp TEXT,
            task_name TEXT,
            status TEXT,
            duration_sec REAL,
            message TEXT
        )
    """)
    
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    for r in workflow_results:
        conn.execute(
            "INSERT INTO pipeline_log VALUES (?, ?, ?, ?, ?)",
            (timestamp, r.task_name, r.status, round(r.duration, 2), r.message)
        )
    conn.commit()
    
    # ログ確認
    log = pd.read_sql("SELECT * FROM pipeline_log ORDER BY rowid", conn)
    print("  【パイプライン実行ログ】")
    print(f"  {log.to_string(index=False)}")
    
    conn.close()
    return log

print("✓ 全タスク関数定義完了")

✓ 全タスク関数定義完了


## 3. DAGの実行

### AirflowでのDAG依存関係
'''python
task_extract >> task_transform >> [task_quality, task_load] >> task_report

### これをPythonで実行:

In [7]:
# DAG実行
print("=" * 60)
print("日経225 ETLパイプライン(Airflow スタイル)")
print(f"スケジュール: 平日毎日 9:00(想定)")
print(f"実行時刻： {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

wf = WorkflowEngine("nikkei225_daily_etl")

# Step1:Extract
extract_result = wf.run_task("1_extract", task_extract, retries=2)

#Extractが失敗したら後続タスクを中断
if extract_result.status== "FAILED":
    print("\nx Extract 失敗のため、パイプラインを中断します")
else:
    df=extract_result.data

    # Step2: Transform
    wf.run_task("2_transform", task_transform, retries=1, df=df)
    
    # Step3: Quality と Loadを並列実行
    wf.run_parallel([
        ("3a_quality_check", task_quality_check, {}),
        ("3b_load_analysis", task_load_analysis, {}),
    ])
    
    # Step4:Report
    wf.run_task("4_report", task_report, retries=1, workflow_results=wf.results)

#　サマリー
all_passed = wf.summary()

日経225 ETLパイプライン(Airflow スタイル)
スケジュール: 平日毎日 9:00(想定)
実行時刻： 2026-04-09 08:27:22

──────────────────────────────────────────────────
[TASK] 1_extract
──────────────────────────────────────────────────
 yfinanceからデータ取得中


[*********************100%***********************]  1 of 1 completed

  1770 行取得
  期間: 2019-01-04 〜 2026-04-09
  ✓ 完了（4.92秒）

──────────────────────────────────────────────────
[TASK] 2_transform
──────────────────────────────────────────────────
  raw_nikkei225: 1770 行
  stg_nikkei225: 作成完了
  int_daily_metrics: 作成完了
  ✓ 完了（0.07秒）

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[PARALLEL] 2 タスクを実行
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

──────────────────────────────────────────────────
[TASK] 3a_quality_check
──────────────────────────────────────────────────
  PASS: NULL チェック
  PASS: 重複チェック
  PASS: 正の値チェック
  PASS: MA20 範囲チェック
  全 4 テスト通過
  ✓ 完了（0.01秒）

──────────────────────────────────────────────────
[TASK] 3b_load_analysis
──────────────────────────────────────────────────
  データ期間: 2019-01-04 〜 2026-04-09
  平均株価: 31,095.61 円
  トレード数: 12
  平均リターン: 1.34%
  p値: 0.745045
  ✓ 完了（0.02秒）

──────────────────────────────────────────────────
[TASK] 4_report
──────────────────────────────────────────────────
  【パイプライン実行ログ】
        run_timestamp

## 4. AirflowのDAGファイル（参考）

実際のAirflowではこのように書く。Day17で実装予定

In [8]:
# Airflow DAGファイルのサンプル（参考表示のみ）
airflow_dag_code = '''
# airflow/dags/nikkei225_daily_etl.py
from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator

default_args = {
    "owner": "data-analytics",
    "retries": 2,
    "retry_delay": timedelta(minutes=5),
    "start_date": datetime(2024, 1, 1),
}

dag = DAG(
    "nikkei225_daily_etl",
    default_args=default_args,
    description="日経225のデイリーETLパイプライン",
    schedule_interval="0 9 * * 1-5",  # 平日毎日9:00
    catchup=False,
)

task_extract   = PythonOperator(task_id="extract",       python_callable=extract,       dag=dag)
task_transform = PythonOperator(task_id="transform",     python_callable=transform,     dag=dag)
task_quality   = PythonOperator(task_id="quality_check", python_callable=quality_check, dag=dag)
task_load      = PythonOperator(task_id="load_analysis", python_callable=load_analysis, dag=dag)
task_report    = PythonOperator(task_id="report",        python_callable=report,        dag=dag)

# DAG の依存関係（これが全て）
task_extract >> task_transform >> [task_quality, task_load] >> task_report
'''

print("【参考：Airflow DAG 定義】")
print(airflow_dag_code)
print("─" * 60)
print("上記の最終行が DAG の依存関係の定義。")
print(">> は「この後に実行」、[] は「並列実行」を意味する。")
print("今回 Python で実装したワークフローと同じ構造。")

【参考：Airflow DAG 定義】

# airflow/dags/nikkei225_daily_etl.py
from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator

default_args = {
    "owner": "data-analytics",
    "retries": 2,
    "retry_delay": timedelta(minutes=5),
    "start_date": datetime(2024, 1, 1),
}

dag = DAG(
    "nikkei225_daily_etl",
    default_args=default_args,
    description="日経225のデイリーETLパイプライン",
    schedule_interval="0 9 * * 1-5",  # 平日毎日9:00
    catchup=False,
)

task_extract   = PythonOperator(task_id="extract",       python_callable=extract,       dag=dag)
task_transform = PythonOperator(task_id="transform",     python_callable=transform,     dag=dag)
task_quality   = PythonOperator(task_id="quality_check", python_callable=quality_check, dag=dag)
task_load      = PythonOperator(task_id="load_analysis", python_callable=load_analysis, dag=dag)
task_report    = PythonOperator(task_id="report",        python_callable=report,        dag=dag)

# DAG の依

## 5.スケジュール設定の解説

Airflowのcron式:
| 式 | 意味 |
|---|---|
| `0 9 * * 1-5` | 平日毎日 9:00 |
| `0 0 * * *` | 毎日 0:00（深夜） |
| `0 9 1 * *` | 毎月1日 9:00 |
| `0 */6 * * *` | 6時間おき |

# 6.まとめ

DAG（Directed Acyclic Graph = 有向非巡回グラフ） とは、
タスクの実行順序と依存関係を定義するネットワーク構造。
Directed = 方向がある、Acyclic = ループしない、Graph = ネットワーク。
Airflow ではこの DAG を定義するだけで、実行順序やエラー時の挙動を自動管理してくれる。

タスクの依存関係とは、「上流が成功しないと下流を実行しない」というルール。
今回の実行では最初に Extract を意図的に失敗させたところ、
Transform 以降が全て中断された。これが DAG の依存関係の効果で、
壊れたデータが下流に流れるのを防ぐ安全装置として機能する。

リトライ機能は、一時的なエラー（ネットワーク不安定など）に対応するための仕組み。
yfinance のようにAPIからデータを取得する Extract は外部要因で失敗しやすいため、
retries=2 で自動的に再試行する設定にしている。

ログ記録は、各タスクの成功/失敗・実行時間・エラー内容をデータベースに保存する仕組み。
pipeline_log テーブルに記録することで、過去の実行履歴を追跡できる。
Airflow では Web UI でこれを可視化でき、ビジネスサイドのメンバーも
ダッシュボード経由でパイプラインの状態を確認できる。

### 出力結果について

全5タスクが成功し、総実行時間は5.02秒。
内訳を見ると Extract が4.92秒（全体の98%）で、Transform は0.07秒、
品質チェックは0.01秒、分析実行が0.01秒、レポートが0.02秒。
ボトルネックは明らかに Extract（yfinance API からのデータ取得）であり、
SQL による加工・テスト・分析はいずれも0.1秒未満で完了している。
Transform と Load の高速化よりも Extract の安定性（リトライ、タイムアウト設定）が重要になりそう。

データは4月9日分まで取得されており、前回（4月2日）から7日分が追加された。
これにより ETL パイプラインが「実行するたびに最新データで更新される」ことを実感。

Airflow を本番導入すれば、cron式（`0 9 * * 1-5`）で平日毎日9時に自動実行され、
Web UI で実行状態をリアルタイムに監視できる。
アラート機能と組み合わせることで、障害発生時の迅速な対応が可能になり、
運用のレジリエンス（回復力）を高められる。

In [6]:
print("✓ Notebook 8完了")

✓ Notebook 8完了
